In [1]:
# Import Libraries
import re
import numpy as np

In [2]:
# Sample Resume Input (Simulating User Upload)
simulated_resume_text = """
John Doe

Experience:
Worked on data analysis projects using Python.
Improved system efficiency.

Education:
Bachelor of Science in Computer Science

Skills:
Python, SQL, Machine Learning
"""

In [3]:
# Select Target Job Role (UC-1)
target_role = "Data Analyst"

In [4]:
# Define Role-Based Keyword Library (Admin Logic)
role_keywords = {
    "Data Analyst": ["python", "sql", "data", "analysis", "visualization", "statistics"],
    "Project Manager": ["planning", "coordination", "stakeholder", "timeline", "budget"]
}

In [5]:
# Resume Class (Core Data Object)
class Resume:
    def __init__(self, text):
        self.raw_text = text
        self.cleaned_text = self.preprocess(text)
        self.sections = self.extract_sections()

    def preprocess(self, text):
        return text.lower()

    def extract_sections(self):
        sections = {
            "experience": "",
            "education": "",
            "skills": ""
        }

        for key in sections:
            pattern = key + r":(.*?)(\n\n|$)"
            match = re.search(pattern, self.cleaned_text, re.DOTALL)
            if match:
                sections[key] = match.group(1).strip()

        return sections

In [6]:
# StructureAnalyzer (UC-4, UC-8)
class StructureAnalyzer:
    def evaluate(self, resume):
        score = 0
        feedback = []

        for section, content in resume.sections.items():
            if content:
                score += 1
            else:
                feedback.append(f"Missing {section} section")

        # Check measurable achievements
        numbers = re.findall(r'\d+', resume.cleaned_text)
        if not numbers:
            feedback.append("No measurable achievements found")

        return score, feedback

In [7]:
# RoleAlignmentAnalyzer (UC-6)
class RoleAlignmentAnalyzer:
    def __init__(self, role_keywords):
        self.role_keywords = role_keywords

    def evaluate(self, resume, role):
        keywords = self.role_keywords.get(role, [])

        matches = [kw for kw in keywords if kw in resume.cleaned_text]
        missing = [kw for kw in keywords if kw not in resume.cleaned_text]

        score = len(matches) / len(keywords) if keywords else 0

        return score, matches, missing

In [8]:
# ATSAnalyzer (UC-5)
class ATSAnalyzer:
    def evaluate(self, resume, role_keywords, role):
        keywords = role_keywords.get(role, [])

        present = [kw for kw in keywords if kw in resume.cleaned_text]
        missing = [kw for kw in keywords if kw not in resume.cleaned_text]

        score = len(present)

        return score, missing

In [9]:
# Weak Phrasing Detection
def detect_weak_phrasing(text):
    weak_words = ["worked on", "responsible for", "helped"]
    found = [word for word in weak_words if word in text]
    return found

In [10]:
# ScoringEngine (UC-7)
class ScoringEngine:
    def compute(self, structure_score, role_score, ats_score):
        final_score = (structure_score * 20) + (role_score * 40) + (ats_score * 10)
        return final_score

In [11]:
# EvaluationReport (UC-7, UC-9)
class EvaluationReport:
    def __init__(self, score, structure_feedback, missing_keywords, weak_phrases):
        self.score = score
        self.structure_feedback = structure_feedback
        self.missing_keywords = missing_keywords
        self.weak_phrases = weak_phrases

    def display(self):
        print("----- Resume Evaluation Report -----")
        print(f"Final Score: {self.score}")
        print(f"Structure Issues: {self.structure_feedback}")
        print(f"Missing Keywords: {self.missing_keywords}")
        print(f"Weak Phrasing: {self.weak_phrases}")

In [12]:
# ResumeEvaluationController
class ResumeEvaluationController:
    def __init__(self, role_keywords):
        self.structure_analyzer = StructureAnalyzer()
        self.role_analyzer = RoleAlignmentAnalyzer(role_keywords)
        self.ats_analyzer = ATSAnalyzer()
        self.scoring_engine = ScoringEngine()
        self.role_keywords = role_keywords

    def evaluate(self, resume_text, role):
        resume = Resume(resume_text)

        # Structure Analysis
        structure_score, structure_feedback = self.structure_analyzer.evaluate(resume)

        # Role Alignment
        role_score, matches, missing_keywords = self.role_analyzer.evaluate(resume, role)

        # ATS Analysis
        ats_score, ats_missing = self.ats_analyzer.evaluate(resume, self.role_keywords, role)

        # Weak phrasing
        weak_phrases = detect_weak_phrasing(resume.cleaned_text)

        # Final scoring
        final_score = self.scoring_engine.compute(
            structure_score,
            role_score,
            ats_score
        )

        # Report
        report = EvaluationReport(
            final_score,
            structure_feedback,
            missing_keywords,
            weak_phrases
        )

        return report

In [13]:
# Run
controller = ResumeEvaluationController(role_keywords)

report = controller.evaluate(simulated_resume_text, target_role)

report.display()

----- Resume Evaluation Report -----
Final Score: 126.66666666666666
Structure Issues: ['No measurable achievements found']
Missing Keywords: ['visualization', 'statistics']
Weak Phrasing: ['worked on']


In [14]:
# Re-Submission
updated_resume = simulated_resume_text + "\nImproved efficiency by 30%"

report = controller.evaluate(updated_resume, target_role)

report.display()

----- Resume Evaluation Report -----
Final Score: 126.66666666666666
Structure Issues: []
Missing Keywords: ['visualization', 'statistics']
Weak Phrasing: ['worked on']


In [15]:
### Unit Test 
def test_structure_analyzer():
    resume_text = "Experience in Python. Education in Computer Science."
    resume = Resume(resume_text)

    analyzer = StructureAnalyzer()
    score, missing = analyzer.evaluate(resume)

    assert score < 1
    assert "skills" in missing
    
def test_role_alignment():
    resume_text = "Experienced in Python and SQL"
    resume = Resume(resume_text)

    analyzer = RoleAlignmentAnalyzer()
    role_keywords = ["python", "sql", "tableau"]

    score, matches, missing = analyzer.evaluate(resume, role_keywords)

    assert "python" in matches
    assert "tableau" in missing
    
def test_scoring_engine():
    scoring = ScoringEngine()
    final_score = scoring.compute(0.8, 0.6, 0.7)

    assert round(final_score, 2) == 0.68
    
def test_controller_evaluate():
    controller = ResumeEvaluationController()

    resume_text = "Experience in Python and SQL. Education in Data Science."
    role_keywords = ["python", "sql", "tableau"]

    report = controller.evaluate(resume_text, role_keywords)

    assert report is not None
    assert report.final_score >= 0